# Check out iSnobal outputs at SNOTEL sites

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import sys
sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from pathlib import PurePath
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

In [ ]:
# Temp test to visualize thp outputs
thp_dir = f'{workdir}/thp'
print(thp_dir)
basin = 'animas'
basindirs = h.fn_list(thp_dir, f'{basin}*/wy*/{basin}*/')
print(basindirs)

In [ ]:
# Test net solar to compare real quick
dt = '20220401'
baseline = h.fn_list(f'{workdir}/animas_100m_isnobal/wy2022/animas_basin_100m', f'run{dt}/net_solar.nc')[0]
print(baseline)
latest = h.fn_list(f'{thp_dir}/animas_hrrr_radiation/wy2022/animas', f'run{dt}/net_solar.nc')[0]
print(latest)


In [ ]:
# Read them in and plot them!
fig, ax = plt.subplots(1, 3, figsize=(15, 6), sharex=True, sharey=True)
da_list = []
for i, fn in enumerate([baseline, latest]):
    ds = xr.open_dataset(fn)
    da = ds.isel(time=18)['net_solar']
    da.plot(ax=ax[i], vmin=0, vmax=800, cmap='magma')
    # Make sure the ax has equal aspect ratio
    ax[i].set_aspect('equal')
    da_list.append(da)

# ax[-1].imshow(da_list[1].values - da_list[0].values)
(da_list[1] - da_list[0]).plot.imshow(ax=ax[-1], cmap='RdBu')
ax[-1].set_aspect('equal')
ax[-1].set_title('Difference (Latest - Baseline)')
# There's way less net solar in the latest version.
# With SPIReS, we would expect more net solar. Interesting.

In [ ]:
# Basin-specific variables
basin = 'animas'

# Select just for all variable output of time decay
basindirs = h.fn_list(workdir, f'{basin}*isnobal/wy*/{basin}*/')
_ = [print(b) for b in basindirs]
# Get the WY from the directory name - assumes there is only one WY per basin right now
WYs = [int(basindir.split('wy')[1][:4]) for basindir in basindirs]
WYs = np.unique(WYs)

if len(WYs) > 1:
    print(f'Multiple water years in {basin} basin: {WYs}')
elif len(WYs) == 0:
    print(f'No water years detected in {basin} basin')

bdx = 1
WY = WYs[bdx]
print(f'{WY} selected')

# Adjust basin dirs based on WY
basindirs = h.fn_list(workdir, f'{basin}*isnobal/wy{WY}/{basin}*/')

# # Figure out filenames
# poly_fn = '/uufs/chpc.utah.edu/common/home/skiles-group1/arobledano_test/br_100m/br_setup/ancillary/br_basin_outline_HUC8_32612.shp'
# print(poly_fn)
poly_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys'
poly_fn = h.fn_list(poly_dir, f'*{basin}*shp')[0]
# SNOTEL sites file
# site_locs_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products//SNOTEL/snotel_sites_32612.json'
# epsg = 32612
# state_abbrev = 'AZ'
site_locs_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products//SNOTEL/snotel_sites_32613.json'
epsg = 32613
state_abbrev = 'CO'
# # Locate basin setup dir
# basin_setupdir = '/uufs/chpc.utah.edu/common/home/skiles-group1/arobledano_test/br_100m/br_setup/'
# print(basin_setupdir)

# Note WY dir
wydir = h.fn_list(workdir, f'{basin}*/wy{WY}/')[0]
print(wydir)

In [ ]:
var = 'depth'
snowvar = 'SNOWDEPTH'
thisvar = 'thickness'

# # Add for SWE and density as well

In [ ]:
# Locate SNOTEL sites within basin
found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=site_locs_fn, buffer=200)

# Get site names and site numbers
sitenames = found_sites['site_name']
sitenums = found_sites['site_num']
print(sitenames)

In [ ]:
ST_arr = [state_abbrev] * len(sitenums)
# gdf_metloom, snotel_dfs, snotel_metadf = proc.get_snotel(sitenums, sitenames, ST_arr, WY=int(WY), epsg=epsg, return_meta=True, snowvar=snowvar)
gdf_metloom, snotel_dfs, snotel_metadf = proc.get_snotel(sitenums, sitenames, ST_arr, WY=[2022, 2023, 2024],
                                                         epsg=epsg, return_meta=True, snowvar=snowvar)
gdf_metloom

In [ ]:
# Locate csvs
thisvar = 'thickness'
# This is for the unified + veg correction runs
csvs = h.fn_list(thp_dir, f'*/*/*{thisvar}*csv')
_ = [print(csv) for csv in csvs]
print('\n')
# This is for the time decay runs and the unified (no veg correction) runs
pre_26_csvs = h.fn_list(workdir, f'{basin}*isnobal/wy*/*{thisvar}*csv')
# _ = [print(csv) for csv in pre_26_csvs]

# Remove the 2021 runs
pre_26_csvs = [csv for csv in pre_26_csvs if '2021' not in csv]
_ = [print(csv) for csv in pre_26_csvs]


In [ ]:
# Combine the csv lists into one list of csvs to read
csvs = csvs + pre_26_csvs
csvs

In [ ]:
df_list = [pd.read_csv(csv, index_col=0) for csv in csvs]
for df in df_list:
    df.index.name = 'Date'
    # Set as DatetimeIndex
    df.index = pd.to_datetime(df.index)

In [ ]:
len(df_list)

In [ ]:
ncols = 2
if len(snotel_dfs.keys()) % 2 == 0:
    nrows = len(snotel_dfs.keys()) // 2
else:
    nrows = len(snotel_dfs.keys()) // 2 + 1
print(nrows, ncols)

In [ ]:
# Create labels based on csv filenames
labels = []
colors = []
for csv in csvs:
    # print(csv)
    # see if this is in the unified + veg correction runs
    thp_match = str(PurePath(csv.split(f'{workdir}/')[1]).parents[-2])
    print(thp_match)
    # split by basin name
    csv = PurePath(csv).stem
    short_name = csv.split(f'{basin}_')[1]
    wy_name = short_name.split('_wy')[1]
    # print(wy_name)
    # split by variable name
    short_name = short_name.split(f'_{thisvar}_')[0]
    # Amend short name if this is from the unified + veg correction runs
    if thp_match == 'thp':
        print('THP')
        short_name = f'{short_name}-THP'
        labels.append(f'HRRR-SPIReS THP')
        colors.append('k')
    elif short_name == 'iSnobal-HRRR':
        labels.append('Baseline')
        colors.append('cornflowerblue')
    else:
        labels.append(f'HRRR-SPIReS - no veg')
        colors.append('coral')

In [ ]:
ylims = (0, 3.5)
if basin == 'blue':
    ylims = (0, 2.5)

xlims = ('2021-10-01', '2024-07-01')


In [ ]:
k

In [ ]:
df_list[0][k]

In [ ]:
# fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3*nrows), sharex=True, sharey=True)
# Do a single row for each site since we're doing multiple years
fig, axes = plt.subplots(len(snotel_dfs.keys()), 1, figsize=(12, 3*len(snotel_dfs.keys())), sharex=True, sharey=True)
# loop through snotel sites
for kdx, k in enumerate(snotel_dfs.keys()):
    ax = axes.flatten()[kdx]
    # Plot snotel data
    snotel_dfs[k]['SNOWDEPTH_m'].plot(label='SNOTEL', ax=ax, c='lightgray', linewidth=0.5, marker='.', markersize=3)
    # Loop through the csv dfs and plot for each snotel site
    for jdx, label in enumerate(labels):
        # print(jdx, k, label)
        if label == 'HRRR-SPIReS THP':
            df_list[jdx][k].plot(label=label, linewidth=0.5,
                        ax=ax, c='k')
        else:
            df_list[jdx][k].plot(label=label, linewidth=1, linestyle='--',
                            ax=ax, c=colors[jdx])
    ax.set_xlim(xlims)
    ax.set_ylim(ylims)
    ax.annotate(k, xy=(0.96, 0.84), xycoords='axes fraction', ha='right', fontsize=11, fontstyle='italic')
    ax.grid('--', alpha=0.2)

# Turn off the last one if that is empty
if len(snotel_dfs.keys()) < len(axes.flatten()):
    ax = axes.flatten()[kdx+1]
    ax.axis('off')
plt.suptitle(f'{basin} SNOTEL comparison with iSnobal {thisvar}', fontsize=12, y=1)

savefig = False
if savefig:
    fig_fn = f'{basin}_wy{WY}_snotelchecks.png'
    fig.savefig(fig_fn, dpi=300, bbox_inches='tight')
plt.tight_layout()

In [ ]:
# only segment out the labels and dfs that do not have veg in them
thp_list = [df for df, label in zip(df_list, labels) if 'THP' in label]
baseline_list = [df for df, label in zip(df_list, labels) if 'Baseline' in label][1:] #drop 2021
print(len(thp_list), len(baseline_list))

In [ ]:
# subtract thp df from each baseline df
diff_dfs = []
for thp_df, baseline_df in zip(thp_list, baseline_list):
    diff_df = thp_df - baseline_df
    diff_dfs.append(diff_df)

In [ ]:
# plot them
for k in snotel_dfs.keys():
    plt.figure()
    for kdx, wy in enumerate([2022, 2023, 2024]):
        diff_dfs[kdx][k].plot(title=k)
    # set the same x and y limits for all
    plt.xlim(xlims)
    plt.ylim(-0.5, 0.25)
    plt.grid(color='lightgray', linestyle='--', alpha=0.5)

In [ ]:
# Check out snow surface temps

# Get the list of snow.nc files for the water year from one of the basin dirs
snow_fn_list = [h.fn_list(basindir, 'run*/snow.nc') for basindir in basindirs]

# Print the length of each list
for snowlist in snow_fn_list:
    print(len(snowlist))

In [ ]:
# Open these lists with the xarray open_mfdataset function
# Drop all the other variables that aren't temperature (keep 'temp*')
snow_ds_list = [xr.open_mfdataset(snowlist, drop_variables=['projection', 'thickness', 'snow_density', 'specific_mass', 'liquid_water', 'thickness_lower', 'water_saturation']) for snowlist in snow_fn_list]

In [ ]:
# Sample datasets for the entire time period at the SNOTEL locations
snow_ds_sampled = [ds['temp_surf'].sel(x=list(gdf_metloom.geometry.x.values),
                             y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in snow_ds_list]

In [ ]:
snow_ds_sampled[0]

In [ ]:
# snow_var_data = xr.concat(snow_var_data, dim='time')

# Plot the time series at the SNOTEL locations
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3*nrows), sharex=True, sharey=True)
for kdx, k in enumerate(snotel_dfs.keys()):
    print(kdx, k)
    ax = axes.flatten()[kdx]
    for jdx, label in enumerate(labels):
        snow_ds_sampled[jdx][:, kdx,kdx].plot(label=label,
                                          ax=ax, c=colors[jdx])
    # ax.set_ylim(ylims)
    ax.annotate(k, xy=(0.96, 0.84), xycoords='axes fraction', ha='right', fontsize=11, fontstyle='italic')
    ax.grid('--', alpha=0.2)
    ax.legend(bbox_to_anchor=(0.7, 0.7))
# Turn off the last one if that is empty
if len(snotel_dfs.keys()) < len(axes.flatten()):
    ax = axes.flatten()[kdx+1]
    ax.axis('off')
# plt.suptitle(f'{basin} WY{WY} SNOTEL comparison with iSnobal {thisvar}', fontsize=12, y=1)

In [ ]:
# Plot the time series at the SNOTEL locations
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3*nrows), sharex=True, sharey=True)
for kdx, k in enumerate(snotel_dfs.keys()):
    print(kdx, k)
    ax = axes.flatten()[kdx]
    for jdx, label in enumerate(labels):
        snow_ds_sampled[jdx][:, kdx,kdx].plot(label=label,
                                          ax=ax, c=colors[jdx])
    ax.annotate(k, xy=(0.96, 0.84), xycoords='axes fraction', ha='right', fontsize=11, fontstyle='italic')
    ax.grid('--', alpha=0.2)
    ax.legend(bbox_to_anchor=(0.7, 0.7))
    ax.set_title('')
# Turn off the last one if that is empty
if len(snotel_dfs.keys()) < len(axes.flatten()):
    ax = axes.flatten()[kdx+1]
    ax.axis('off')
# plt.suptitle(f'{basin} WY{WY} SNOTEL comparison with iSnobal {thisvar}', fontsize=12, y=1)